In [26]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))


In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import DATASETS_DIR
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

In [28]:
df = pd.read_csv(DATASETS_DIR / "regression.csv", index_col=0)

In [29]:
df.drop(columns=['time', 'pressure_air','d_depth', 'd_time', 'n/p'], inplace=True)

In [30]:
df

,depth,angle,rotation,pressure_axis,pressure_rotation,well_id,speed,log_speed,cluster_baseline,cluster_no_depth
0,4.6965,93.4,104.892,12781,13144,25512,0.020200,0.019999,5,13
1,4.8783,93.0,101.430,18971,18936,25512,0.036360,0.035715,12,5
2,5.0298,92.1,101.430,20153,18184,25512,0.030300,0.029850,12,12
3,5.1813,93.0,102.714,20162,17445,25512,0.030300,0.029850,12,12
4,5.3934,94.3,47.724,1233,10808,25512,0.007856,0.007825,1,11
...,...,...,...,...,...,...,...,...,...,...
160891,17.6952,85.5,103.008,22690,16457,31958,0.008264,0.008230,11,27
160892,17.7861,87.0,101.826,22696,17037,31958,0.015150,0.015036,11,27
160893,17.8164,86.7,102.120,22682,16439,31958,0.006060,0.006042,11,27
160894,17.8770,93.2,100.044,22661,18501,31958,0.012120,0.012047,15,5


In [31]:
features = ['depth', 'angle', 'rotation', 'pressure_axis', 'pressure_rotation', 'cluster_no_depth']
X = df[features]
y = df['log_speed']  # логарифмируемый target
groups = df['well_id']  # для GroupKFold

In [32]:
gkf = GroupKFold(n_splits=5)
rmse_list = []

for train_idx, val_idx in gkf.split(X, y, groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    dt = DecisionTreeRegressor(random_state=21)
    dt.fit(X_train, y_train)
    
    y_pred = dt.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    rmse_list.append(rmse)

print("RMSE per fold:", rmse_list)
print("Mean RMSE:", np.mean(rmse_list))

RMSE per fold: [np.float64(0.02977708232763976), np.float64(0.026681861923857648), np.float64(0.026715528314613074), np.float64(0.023610570696364533), np.float64(0.027685917080967865)]
Mean RMSE: 0.026894192068688576


In [ ]:
rmse_list_gb = []

for train_idx, val_idx in tqdm(gkf.split(X, y, groups)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    gb = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=21
    )
    gb.fit(X_train, y_train)
    
    y_pred = gb.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    rmse_list_gb.append(rmse)



5it [01:20, 16.11s/it]

Gradient Boosting RMSE per fold: [np.float64(0.018713850135358128), np.float64(0.01916437386619005), np.float64(0.018887143399824505), np.float64(0.01684655431000993), np.float64(0.020319789046117847)]
Mean RMSE: 0.018786342151500095
